# Cross-Validation Techniques

A comprehensive guide to model validation strategies for reliable performance estimation.

## Topics Covered
1. Hold-Out Validation
2. K-Fold Cross-Validation
3. Stratified K-Fold
4. Leave-One-Out (LOO)
5. Repeated K-Fold
6. Time Series Split
7. Group K-Fold
8. Nested Cross-Validation (for hyperparameter tuning)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    LeaveOneOut, RepeatedKFold, TimeSeriesSplit,
    GroupKFold, cross_val_score, GridSearchCV,
    cross_validate
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load dataset
iris = load_iris()
X, y = iris.data, iris.target
print(f"Dataset shape: {X.shape}")
print(f"Classes: {np.unique(y)} (counts: {np.bincount(y)})")

## 1. Hold-Out Validation (Train-Test Split)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
score = model.score(X_test, y_test)
print(f"Hold-Out Accuracy: {score:.4f}")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 2. K-Fold Cross-Validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
print(f"K-Fold (k=5) Scores: {scores}")
print(f"Mean: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Visualize splits
fig, axes = plt.subplots(1, 5, figsize=(20, 3))
for i, (train_idx, test_idx) in enumerate(kf.split(X)):
    mask = np.zeros(len(X))
    mask[test_idx] = 1
    axes[i].bar(range(len(X)), mask, width=1, color=['steelblue' if m == 0 else 'coral' for m in mask])
    axes[i].set_title(f'Fold {i+1}\nAcc={scores[i]:.3f}')
    axes[i].set_ylabel('Test=1')
plt.suptitle('K-Fold Split Visualization (Blue=Train, Red=Test)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Stratified K-Fold
Ensures each fold has the same class distribution as the full dataset.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print(f"Stratified K-Fold Scores: {scores}")
print(f"Mean: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Show class distribution per fold
print("\nClass distribution per fold:")
for i, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    train_dist = np.bincount(y[train_idx])
    test_dist = np.bincount(y[test_idx])
    print(f"  Fold {i+1}: Train={train_dist}, Test={test_dist}")

## 4. Leave-One-Out Cross-Validation (LOO-CV)

In [ ]:
loo = LeaveOneOut()
model_simple = DecisionTreeClassifier(random_state=42)
scores = cross_val_score(model_simple, X, y, cv=loo, scoring='accuracy')

print(f"LOO-CV: {len(scores)} folds (one per sample)")
print(f"Mean Accuracy: {scores.mean():.4f}")
print(f"Errors: {(scores == 0).sum()} out of {len(scores)}")

## 5. Repeated K-Fold

In [ ]:
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
scores = cross_val_score(model, X, y, cv=rkf, scoring='accuracy')

print(f"Repeated K-Fold (5-fold x 10 repeats = {len(scores)} evaluations)")
print(f"Mean: {scores.mean():.4f} (+/- {scores.std():.4f})")

plt.figure(figsize=(10, 4))
plt.boxplot(scores.reshape(10, 5).T, labels=[f'Rep {i+1}' for i in range(10)])
plt.ylabel('Accuracy')
plt.title('Repeated K-Fold: Accuracy Distribution per Repeat')
plt.axhline(y=scores.mean(), color='r', linestyle='--', label=f'Overall Mean={scores.mean():.4f}')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Time Series Split
For temporal data where future data shouldn't leak into training.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

# Visualize time series splits
fig, ax = plt.subplots(figsize=(12, 5))
n_samples = len(X)
for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    ax.barh(i, len(train_idx), left=0, color='steelblue', alpha=0.7)
    ax.barh(i, len(test_idx), left=len(train_idx), color='coral', alpha=0.7)
    ax.text(len(train_idx)/2, i, f'Train: {len(train_idx)}', ha='center', va='center', fontsize=9)
    ax.text(len(train_idx)+len(test_idx)/2, i, f'Test: {len(test_idx)}', ha='center', va='center', fontsize=9)

ax.set_yticks(range(5))
ax.set_yticklabels([f'Split {i+1}' for i in range(5)])
ax.set_xlabel('Sample Index')
ax.set_title('Time Series Split (Train grows, Test slides forward)')
plt.tight_layout()
plt.show()

## 7. Nested Cross-Validation
Unbiased estimate of model performance with hyperparameter tuning.

In [ ]:
# Outer CV: evaluates model performance
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Inner CV: tunes hyperparameters
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Pipeline with scaling + SVM
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])

param_grid = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['rbf', 'linear'],
    'svm__gamma': ['scale', 'auto']
}

# GridSearchCV as the inner loop
grid_search = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring='accuracy', n_jobs=-1)

# Outer cross-validation
nested_scores = cross_val_score(grid_search, X, y, cv=outer_cv, scoring='accuracy')

print(f"Nested CV Scores: {nested_scores}")
print(f"Mean: {nested_scores.mean():.4f} (+/- {nested_scores.std():.4f})")

## 8. Comparing Multiple Models with Cross-Validation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f"{name:25s} | Mean={scores.mean():.4f} | Std={scores.std():.4f}")

# Box plot comparison
plt.figure(figsize=(12, 6))
plt.boxplot(results.values(), labels=results.keys())
plt.ylabel('Accuracy')
plt.title('Model Comparison via 10-Fold Stratified Cross-Validation')
plt.xticks(rotation=15)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Method | Best For | Pros | Cons |
|--------|----------|------|------|
| Hold-Out | Large datasets | Fast | High variance |
| K-Fold | General use | Balanced | Moderate compute |
| Stratified K-Fold | Imbalanced classes | Preserves distribution | Moderate compute |
| LOO-CV | Small datasets | Low bias | Very slow |
| Repeated K-Fold | Robust estimation | Low variance | Slow |
| Time Series Split | Temporal data | Respects time order | Increasing train size |
| Nested CV | With hyperparameter tuning | Unbiased estimation | Computationally expensive |